In [1]:
# Proof of concept and error-check for EMMA uncertainty calc module
# which is based on David Genereux's 1998 paper "Uncertainty in Tracer-Based Hydrograph Separations".
# That paper uses a streamwater sample with 18O and Cl tracer data from Bazemore et al., 1994 to demonstrate
# a 3-component separation error propagation.
# Here, I make confirm I get the same uncertainties as Genereux 1998 using his Table 3 input (from Bazemore) and my module.

import importlib
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Set up home path and working repo dir 
HOME = Path.home()
repo_dir = HOME / "OneDrive/git-repos/LCBP-interannual-EMMAs"

if str(repo_dir) not in sys.path:
    sys.path.append(str(repo_dir))

# Import EMMA module
import EMMA.event_emma as em  
import EMMA.event_pca_error as ep
import EMMA.uncertainty_propagation as up
importlib.reload(ep)
importlib.reload(em)
importlib.reload(up)

# Define data directory using relative path logic
data_dir = repo_dir / "Data/GrabSample_data"

# Load the full RI25 dataset
df = pd.read_csv(data_dir / "RI23-IC-ICP-isotope-toc-joined.csv")

############################## FUNCTION TESTING ##############################
# 1. Recreate the Bazemore et al. (1994) sampled from Genereux (1998) Table 3 dataset
# End-members (Event water, Preevent soil water, Preevent groundwater)
em_data = {
    "Type": ["Event water", "Preevent soil water", "Preevent groundwater"],
    "Cl": [4.0, 26.6, 26.0],
    "d18O": [-8.1, -6.1, -7.6],
}
em_grouped = pd.DataFrame(em_data)

# Recreate the raw end-member dataframe to calculate/pass standard deviations.
# Since the py uncertainty function uses `em_raw` to extract standard deviations via `.std()`,
# we will construct a mock dataframe with exactly 2 samples per end-member
# that yields the exact mean and STDV values from Table 3.
em_raw_list = []
for idx, row in em_grouped.iterrows():
    m_cl, sd_cl = row["Cl"], [0.8, 8.3, 3.6][idx]
    m_o, sd_o = row["d18O"], [0.16, 0.20, 0.26][idx]

    # Two points spaced by sqrt(2)*SD will yield exactly the mean and target SD
    offset_cl = sd_cl / np.sqrt(2)
    offset_o = sd_o / np.sqrt(2)

    em_raw_list.append(
        {
            "Type": row["Type"],
            "Cl": m_cl + offset_cl,
            "d18O": m_o + offset_o,
            "Sample ID": f"{row['Type']}_1",
        }
    )
    em_raw_list.append(
        {
            "Type": row["Type"],
            "Cl": m_cl - offset_cl,
            "d18O": m_o - offset_o,
            "Sample ID": f"{row['Type']}_2",
        }
    )

em_raw = pd.DataFrame(em_raw_list)

# 2. Create the Streamwater mixture sample from the bottom row of Table 3
# Mean stream water Cl = 23, d18O = -6.7.
# Standard analytical precision (analytical error of the single sample): Cl = 0.7, d18O = 0.19
stream_df = pd.DataFrame(
    [
        {
            "Sample ID": "Stream_Sample_1",
            "Datetime": "1994-01-01 12:00:00",
            "Cl": 23.0,
            "d18O": -6.7,
        }
    ]
)

# Analytical SD dictionary for the stream water measurement
analytical_sd = {"Cl": 0.7, "d18O": 0.19}

# 3. Run the uncertainty propagation
tracers = ["Cl", "d18O"]

# Calculate the baseline fractions first to see the source breakdown
em_means_matrix = em_grouped.set_index("Type")[tracers].values
stream_mixture = stream_df[tracers].values[0]
calculated_fractions = up.solve_fractions(stream_mixture, em_means_matrix)

# Run the propagation engine
uncertainty_df = up.propagate_genereux_uncertainty(
    stream_df=stream_df,
    em_grouped=em_grouped,
    em_raw=em_raw,
    tracers=tracers,
    analytical_sd=analytical_sd,
)

# 4. Display Results
print("--- BAZEMORE ET AL. (1994) VERIFICATION ---")
print(f"Tracers used: {tracers}\n")
for i, source in enumerate(em_grouped["Type"]):
    frac = calculated_fractions[i]
    unc = uncertainty_df[f"{source}_Uncertainty_1sig"].values[0]
    print(f"{source:<22}: Fraction = {frac:.3f} | 1-Sigma Uncertainty (W_f) = ±{unc:.3f}")

--- BAZEMORE ET AL. (1994) VERIFICATION ---
Tracers used: ['Cl', 'd18O']

Event water           : Fraction = 0.154 | 1-Sigma Uncertainty (W_f) = ±0.252
Preevent soil water   : Fraction = 0.651 | 1-Sigma Uncertainty (W_f) = ±0.181
Preevent groundwater  : Fraction = 0.194 | 1-Sigma Uncertainty (W_f) = ±0.375
